# 4.14 Kernel PCA

Kernel PCA finds nonlinear structure by applying PCA to a centered similarity matrix.

Part 4 moves from prediction with labels to discovery without labels. Vectors, covariance, kernels, and matrix factorization become the language for hidden low-dimensional structure. Geometry supplies distances, neighborhoods, projections, and matrix shapes; optimization decides which structure is kept.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.decomposition import FactorAnalysis
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Lasso
from sklearn.manifold import trustworthiness
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

np.random.seed(7)


def dimred_ladder():
    """D1..D5 dimensionality-reduction ladder. Returns [(name, X, y), ...] of rising ambient dim.

    2-D toy -> 3-D swiss-roll-ish -> digits(64-D) -> the same with noise dims -> a wide feature set.
    y is a color/label for visualization only.
    """
    rungs = []
    rng = np.random.default_rng(3)

    t = np.linspace(0, 4, 120)
    x1 = np.column_stack([t, 0.5 * t + rng.normal(0, 0.05, 120)])
    rungs.append(("D1 near-1-D line in 2-D", x1, t))

    tt = np.linspace(0, 3 * np.pi, 200)
    x2 = np.column_stack([tt * np.cos(tt), 8 * rng.random(200), tt * np.sin(tt)])
    rungs.append(("D2 swiss-roll (3-D)", x2, tt))

    digits = load_digits()
    rungs.append(("D3 digits (real, 64-D)", digits.data / 16.0, digits.target))

    xn = np.hstack([digits.data / 16.0, rng.normal(0, 1, size=(digits.data.shape[0], 32))])
    rungs.append(("D4 digits + 32 noise dims", xn, digits.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (30-D)", bc.data, bc.target))

    return rungs



def scale_for_geometry(X):
    scaler = StandardScaler()
    return scaler.fit_transform(X)


def pad_to_2d(Z):
    if Z.shape[1] >= 2:
        return Z[:, :2]
    zeros = np.zeros((Z.shape[0], 1))
    return np.hstack([Z, zeros])


def reconstruction_rmse(X, X_hat):
    return float(np.sqrt(np.mean((X - X_hat) ** 2)))


def plot_embedding_panels(results, metric_label):
    fig = plt.figure(figsize=(15, 6))
    for idx, item in enumerate(results):
        ax = fig.add_subplot(2, 5, idx + 1)
        Z2 = pad_to_2d(item["Z"])
        ax.scatter(Z2[:, 0], Z2[:, 1], c=item["y"], s=8, cmap="viridis", alpha=0.8)
        ax.set_title(item["name"].split("(")[0].strip(), fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    ax = fig.add_subplot(2, 1, 2)
    xs = np.arange(1, len(results) + 1)
    ys = [item["metric"] for item in results]
    ax.plot(xs, ys, marker="o")
    ax.set_xticks(xs)
    ax.set_xticklabels(["D" + str(i) for i in xs])
    ax.set_ylabel(metric_label)
    ax.set_title(metric_label + " vs ladder rung")
    fig.tight_layout()
    return fig


## The concept, built once: local influence from an RBF kernel

The lesson's kernel weight calculation is $$p_i=\frac{\exp(-d_i^2/2h^2)}{\sum_j\exp(-d_j^2/2h^2)}$$.
With $h=1$ and distances $[1,0,1]$, weights are $[0.607,1.000,0.607]$, their sum is $2.213$, and the middle influence is $1/2.213=0.452$.

In [ ]:

distances = np.array([1.0, 0.0, 1.0])
h = 1.0
weights = np.exp(-(distances ** 2) / (2 * h ** 2))
weight_sum = weights.sum()
influence = weights[1] / weight_sum

assert np.allclose(np.round(weights, 3), [0.607, 1.000, 0.607])
assert np.isclose(np.round(weight_sum, 3), 2.213)
assert np.isclose(np.round(influence, 3), 0.452)

print("weights", np.round(weights, 3))
print("weight sum", np.round(weight_sum, 3))
print("middle influence", np.round(influence, 3))


Now implement real Kernel PCA: build an RBF kernel, double-center it, eigendecompose the centered kernel, and scale eigenvectors into embedding coordinates.

In [ ]:

def kernel_pca_method(X, gamma=0.2, n_components=2):
    X = np.asarray(X, dtype=float)
    sq_dists = pairwise_distances(X, metric="sqeuclidean")
    K = np.exp(-gamma * sq_dists)
    n = K.shape[0]
    one = np.ones((n, n)) / n
    K_centered = K - one @ K - K @ one + one @ K @ one
    eigenvalues, eigenvectors = np.linalg.eigh(K_centered)
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = np.maximum(eigenvalues[order], 0.0)
    eigenvectors = eigenvectors[:, order]
    safe_values = np.maximum(eigenvalues[:n_components], 1e-12)
    Z = eigenvectors[:, :n_components] * np.sqrt(safe_values)
    return Z, K_centered, eigenvalues

kernel_toy = np.array([
    [-1.0, 0.0],
    [0.0, 0.0],
    [1.0, 0.0],
    [0.0, 1.0],
])
Z_kernel, K_centered, eigenvalues = kernel_pca_method(kernel_toy, gamma=0.5, n_components=2)
assert Z_kernel.shape == (4, 2)
assert K_centered.shape == (4, 4)
print("kernel toy embedding shape", Z_kernel.shape)
print("top eigenvalues", np.round(eigenvalues[:2], 3))


## The dataset ladder

Kernel PCA uses distances, so each rung is standardized before the kernel is built.

In [ ]:

rungs = dimred_ladder()
for idx, (name, X, y) in enumerate(rungs, start=1):
    y_array = np.asarray(y)
    unique_preview = np.unique(y_array[: min(len(y_array), 200)])[:8]
    print(f"D{idx}: {name}")
    print(f"  X shape: {X.shape}")
    print(f"  y preview: {unique_preview}")
    print(f"  first row: {np.round(X[0, : min(6, X.shape[1])], 3)}")


## Run the same Kernel PCA method across D1-D5

In [ ]:

results = []
for name, X, y in dimred_ladder():
    X_scaled = scale_for_geometry(X)
    Z, K_centered, eigenvalues = kernel_pca_method(X_scaled, gamma=0.05, n_components=2)
    tw = float(trustworthiness(X_scaled, Z, n_neighbors=10))
    results.append({"name": name, "Z": Z, "y": y, "metric": tw})

print("rung | trustworthiness")
for idx, item in enumerate(results, start=1):
    print(f"D{idx} | {item['metric']:.3f} | {item['name']}")


## Results visualization

The panels show nonlinear Kernel PCA coordinates; the summary curve tracks trustworthiness, a local-neighborhood preservation metric.

In [ ]:

fig = plot_embedding_panels(results, "trustworthiness")
plt.show()


## Pitfall on D5: kernel bandwidth collapse

If $\gamma$ is too small, all points look almost equally near. If $\gamma$ is too large, almost all points look far. A sweep makes the choice auditable.

In [ ]:

name, X_d5, y_d5 = dimred_ladder()[-1]
X_d5_scaled = scale_for_geometry(X_d5)
gammas = [0.0001, 0.001, 0.01, 0.05, 0.2, 1.0]
scores = []
for gamma in gammas:
    Z, K_centered, eigenvalues = kernel_pca_method(X_d5_scaled, gamma=gamma, n_components=2)
    score = float(trustworthiness(X_d5_scaled, Z, n_neighbors=10))
    scores.append(score)

best_idx = int(np.argmax(scores))
print("gamma sweep")
for gamma, score in zip(gammas, scores):
    print(f"  gamma={gamma}: trustworthiness={score:.3f}")
print("best gamma", gammas[best_idx])


## Evaluate it + Practice

- Track the ladder metric: trustworthiness. Compare it with a no-skill baseline such as using the first two raw scaled columns or the input mean reconstruction.
- Sanity check shapes: D1-D5 should all return a two-column visualization embedding and a scalar metric.
- Ablation: skip the gamma sweep and trust one arbitrary bandwidth; the metric should get worse or the visualization should lose structure.
- Failure signals: tiny hyperparameter changes flip the pattern, D5 improves only on the training objective, or one raw-scale feature dominates.
- Stability check: rerun with a nearby seed or hyperparameter and compare reconstruction, trustworthiness, or neighbor preservation rather than component names.

Practice 1: Change the number of components or atoms and redraw the metric curve.

Practice 2: Compare Kernel PCA with linear PCA on D2 swiss-roll.

Practice 3: Plot the D1 centered kernel matrix as an image.